## Data Preparation


In [1]:
# Transforming Coordinates
import pandas as pd

# Loading the I-395 boundary file
boundaries_df = pd.read_csv("I395_boundaries.csv")

# Computing H and applying transformation
y_coords = boundaries_df["y"].values
H = y_coords.max() - y_coords.min()
pixel_to_meter = 0.3

# Applying transformations 
transformed_df = boundaries_df.copy()
for col in ["x1", "x2", "x3", "x4", "x5"]:
    transformed_df[col] = transformed_df[col] * pixel_to_meter
transformed_df["y"] = (H - transformed_df["y"]) * pixel_to_meter      # bottom left reference

# Saving to file
transformed_df.to_csv("Transformed_I-395_Boundaries.csv", index=False)


In [2]:
# Transforming the coordinates for the entire dataset so the reference (0,0) is bottom-left

import pandas as pd

# Loading dataset
tgsim = pd.read_csv("Third_Generation_Simulation_Data__TGSIM__I-395_Trajectories.csv")

# Converting H from the previous code cell to meters
H = 2530                 # difference between min and max y coordinate 
pixel_to_meter = 0.3     # from documentation
H_meters = H * pixel_to_meter

# Applying transformations to y-coordinates, y-speeds and y-accelerations
tgsim["yloc_kf"] = H_meters - tgsim["yloc_kf"]

# Saving transformed dataset
tgsim.to_csv("Transformed_TGSIM_I-395.csv", index=False)

In [3]:
# Smoothing and Calculation of Variables - I-395 Dataset

import pandas as pd
import numpy as np
from scipy.ndimage import gaussian_filter1d

# Loading and sorting dataset
df = pd.read_csv("Transformed_TGSIM_I-395.csv")
df = df.sort_values(by=["id", "time"]).reset_index(drop=True)

# Sampling interval (10 Hz)
dt = 0.1  # seconds per frame

# Helper: Gaussian smoothing
def gaussian_smooth(series, sigma=1.0):
    """Apply Gaussian filter with nearest-edge handling."""
    return pd.Series(
        gaussian_filter1d(series.to_numpy(), sigma=sigma, mode="nearest"),
        index=series.index
    )

# Function to compute v_x, v_y, a_x, a_y
def compute_kinematics(group, sigma=1.0):
    # Smooth positions first
    x_smooth = gaussian_filter1d(group["xloc_kf"].values, sigma=sigma, mode="nearest")
    y_smooth = gaussian_filter1d(group["yloc_kf"].values, sigma=sigma, mode="nearest")

    # First derivatives → velocity
    vx = np.gradient(x_smooth, dt)
    vy = np.gradient(y_smooth, dt)

    # Second derivatives → acceleration
    ax = np.gradient(vx, dt)
    ay = np.gradient(vy, dt)

    # Assign results
    group["xloc_kf_smooth"] = x_smooth
    group["yloc_kf_smooth"] = y_smooth
    group["speed_kf_x"] = vx
    group["speed_kf_y"] = vy
    group["acceleration_kf_x"] = ax
    group["acceleration_kf_y"] = ay

    # Derived magnitudes
    group["speed_kf_smooth"] = np.sqrt(vx**2 + vy**2)
    # Signed acceleration (projection of acceleration on velocity direction)
    speed = np.hypot(vx, vy)
    signed_acc = np.zeros(len(group))

    for i in range(len(group)):
        if speed[i] > 0:
            signed_acc[i] = (vx[i]*ax[i] + vy[i]*ay[i]) / speed[i]
        else:
            signed_acc[i] = 0.0

    group["acceleration_kf"] = signed_acc
    return group

# Apply per agent (id)
df = df.groupby("id", group_keys=False).apply(lambda g: compute_kinematics(g, sigma=1.0))

# Saving final dataset
df.to_csv("Transformed_TGSIM_I-395_smoothed.csv", index=False)

print("Saved dataset with smoothed (x, y), velocity, and acceleration components.")

C:\Users\msela\AppData\Local\Temp\ipykernel_33272\1325229176.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("id", group_keys=False).apply(lambda g: compute_kinematics(g, sigma=1.0))


Saved dataset with smoothed (x, y), velocity, and acceleration components.


Index(['id', 'time', 'xloc_kf', 'yloc_kf', 'lane_kf', 'speed_kf',
       'acceleration_kf', 'length_smoothed', 'width_smoothed',
       'type_most_common', 'xloc_kf_smooth', 'yloc_kf_smooth', 'speed_kf_x',
       'speed_kf_y', 'acceleration_kf_x', 'acceleration_kf_y',
       'speed_kf_smooth'],
      dtype='object')

## Detecting Surrounding Agents for Behavior and Context Variable Computation

In [ ]:
import pandas as pd
import numpy as np

# ======================================
# Load full smoothed dataset
# ======================================
print("Loading dataset...")
tgsim = pd.read_csv("Transformed_TGSIM_I-395_smoothed.csv")

# Define window id (1000-second bins)
tgsim["window_id"] = (tgsim["time"] // 1000).astype(int)

# Tesla zones
tesla_zones = {
    "Wide Fwd": {"radius": 60, "angle": 120, "from": 300, "to": 60},
    "Main Fwd": {"radius": 150, "angle": 45, "from": 337.5, "to": 22.5},
    "Narrow Fwd": {"radius": 250, "angle": 35, "from": 342.5, "to": 17.5},
    "Side Fwd L": {"radius": 80, "angle": 90, "from": 25, "to": 115},
    "Side Fwd R": {"radius": 80, "angle": 90, "from": 245, "to": 335},
    "Rear": {"radius": 50, "angle": 135, "from": 112.5, "to": 247.5},
    "Side Rear L": {"radius": 100, "angle": 75, "from": 107.5, "to": 182.5},
    "Side Rear R": {"radius": 100, "angle": 75, "from": 177.5, "to": 252.5}
}

def normalize_angle(angle):
    return angle % 360

def rotate_zones(zones, heading):
    rotated = {}
    for name, params in zones.items():
        rotated[name] = {
            "radius": params["radius"],
            "from": normalize_angle(params["from"] + heading),
            "to": normalize_angle(params["to"] + heading)
        }
    return rotated

def detect_zones(rel_angle, distance, zones):
    hits = []
    rel_angle = normalize_angle(rel_angle)
    for zone, params in zones.items():
        f, t = params["from"], params["to"]
        if f > t:
            within = (rel_angle >= f or rel_angle <= t)
        else:
            within = (f <= rel_angle <= t)
        if within and distance <= params["radius"]:
            hits.append(zone)
    return hits

# Compute heading once
print("Computing headings...")
tgsim["heading"] = np.degrees(np.arctan2(
    tgsim["speed_kf_y"],
    tgsim["speed_kf_x"]
))

tgsim["heading"] = tgsim.groupby("id")["heading"].transform(lambda x: x.ffill().bfill())
tgsim["heading"] = tgsim["heading"].apply(normalize_angle)

# ======================================
# Process each window separately
# ======================================
for window in sorted(tgsim["window_id"].unique()):

    print(f"Processing window {window}")

    sub = tgsim[tgsim["window_id"] == window]

    agent_data = sub[sub["type_most_common"].isin([1, 2, 3, 4])]

    detection_details = []

    for timestep in sorted(sub["time"].unique()):

        hdv_positions = sub[
            (sub["time"] == timestep) &
            (sub["type_most_common"] == 1)
        ]

        agent_positions = agent_data[agent_data["time"] == timestep]

        # Convert to numpy once
        hdv_array = hdv_positions[[
            "id", "xloc_kf_smooth", "yloc_kf_smooth", "heading"
        ]].to_numpy()
        
        agent_array = agent_positions[[
            "id", "xloc_kf_smooth", "yloc_kf_smooth"
        ]].to_numpy()
        
        if len(hdv_array) == 0 or len(agent_array) == 0:
            continue
        
        hdv_ids = hdv_array[:, 0].astype(int)
        hdv_x = hdv_array[:, 1]
        hdv_y = hdv_array[:, 2]
        hdv_heading = hdv_array[:, 3]
        
        agent_ids = agent_array[:, 0].astype(int)
        agent_x = agent_array[:, 1]
        agent_y = agent_array[:, 2]
        
        # Pairwise dx, dy (broadcasting)
        dx = agent_x[None, :] - hdv_x[:, None]
        dy = agent_y[None, :] - hdv_y[:, None]
        
        distance = np.sqrt(dx**2 + dy**2)
        
        # Skip near-zero distances
        valid_mask = distance >= 0.01
        
        rel_angle = np.degrees(np.arctan2(dy, dx))
        rel_angle = normalize_angle(rel_angle)
        
        # Loop only over HDVs (small loop)
        for i in range(len(hdv_ids)):
        
            rotated = rotate_zones(tesla_zones, hdv_heading[i])
        
            for j in np.where(valid_mask[i])[0]:
        
                zones_hit = detect_zones(rel_angle[i, j], distance[i, j], rotated)
        
                for zone in zones_hit:
                    detection_details.append({
                        "time": timestep,
                        "hdv_id": int(hdv_ids[i]),
                        "agent_id": int(agent_ids[j]),
                        "zone": zone,
                        "distance": distance[i, j]
                    })

    pd.DataFrame(detection_details).to_csv(
        f"HDV_Detection_Details_I-395_window_{window}.csv",
        index=False
    )

    print(f"Window {window} saved.")

print("All windows completed.")

## Defining Leader and Adding Behavior Variables

In [4]:
import pandas as pd
import numpy as np
import glob

# ===============================
# Load smoothed dataset once
# ===============================
df = pd.read_csv("Transformed_TGSIM_I-395_smoothed.csv")

# ===============================
# Base HDV dataset (RENAMED columns kept)
# ===============================
hdv_base = df[df["type_most_common"] == 1][[
    "id", "time",
    "lane_kf",
    "speed_kf_smooth",
    "xloc_kf_smooth",
    "yloc_kf_smooth",
    "speed_kf_x",
    "speed_kf_y",
    "acceleration_kf",
    "acceleration_kf_x",
    "acceleration_kf_y"
]].rename(columns={
    "id": "hdv_id",
    "lane_kf": "lane_hdv",
    "speed_kf_smooth": "v_hdv",
    "xloc_kf_smooth": "xloc_hdv",
    "yloc_kf_smooth": "yloc_hdv",
    "speed_kf_x": "vx_hdv",
    "speed_kf_y": "vy_hdv",
    "acceleration_kf": "a_hdv",
    "acceleration_kf_x": "ax_hdv",
    "acceleration_kf_y": "ay_hdv"
})

# ===============================
# Add jerk (backward difference)
# ===============================
TIME_STEP = 0.1

hdv_base = hdv_base.sort_values(["hdv_id", "time"])

# Backward jerk
hdv_base["jerk_hdv"] = (
    hdv_base.groupby("hdv_id")["a_hdv"]
    .diff() / TIME_STEP
)

# Forward jerk for first timestep of each vehicle
forward_jerk = (
    hdv_base.groupby("hdv_id")["a_hdv"]
    .diff(-1) / -TIME_STEP
)

# Replace NaNs (first timestep) with forward jerk
hdv_base["jerk_hdv"] = hdv_base["jerk_hdv"].fillna(forward_jerk)

# Optional: if any remain (single-frame vehicles)
hdv_base["jerk_hdv"] = hdv_base["jerk_hdv"].fillna(0)

# ===============================
# Leader info (RENAMED)
# ===============================
leader_info = df[[
    "id", "time",
    "lane_kf",
    "speed_kf_smooth",
    "type_most_common",
    "xloc_kf_smooth",
    "yloc_kf_smooth",
    "speed_kf_x",
    "speed_kf_y",
    "acceleration_kf",
    "acceleration_kf_x",
    "acceleration_kf_y"
]].rename(columns={
    "id": "agent_id",
    "lane_kf": "lane_lead",
    "speed_kf_smooth": "v_lead",
    "type_most_common": "leader_type",
    "xloc_kf_smooth": "xloc_lead",
    "yloc_kf_smooth": "yloc_lead",
    "speed_kf_x": "vx_lead",
    "speed_kf_y": "vy_lead",
    "acceleration_kf": "a_lead",
    "acceleration_kf_x": "ax_lead",
    "acceleration_kf_y": "ay_lead"
})

# ===============================
# Lane compatibility
# ===============================
lane_map_i395 = {-1:[-1],-2:[-2],-3:[-3],-4:[-4],-5:[-5]}

lane_pairs = pd.DataFrame(
    [(k, v) for k, vals in lane_map_i395.items() for v in vals],
    columns=["lane_hdv", "lane_lead"]
)

forward_zones = {"Main Fwd", "Narrow Fwd"}

# ===============================
# Process detection windows
# ===============================
leader_results = []

detection_files = sorted(glob.glob("HDV_Detection_Details_I-395_window_*.csv"))

for file in detection_files:
    det = pd.read_csv(file)

    det = det[det["zone"].isin(forward_zones)]
    det = det[det["hdv_id"] != det["agent_id"]]

    det = det.merge(hdv_base, on=["hdv_id", "time"], how="left")
    
    det = det.merge(leader_info, on=["agent_id", "time"], how="left")

    # Keep only valid leader vehicle types
    valid_leader_types = {1, 2, 3, 4}
    det = det[det["leader_type"].isin(valid_leader_types)]
    
    det = det.merge(lane_pairs, on=["lane_hdv", "lane_lead"], how="inner")

    det = det.sort_values("distance")
    leaders = det.groupby(["hdv_id", "time"], as_index=False).first()

    leaders["headway_s"] = leaders["distance"]
    leaders["delta_v"] = leaders["v_hdv"] - leaders["v_lead"]
    leaders["headway_t"] = np.where(
        leaders["v_hdv"] > 0,
        leaders["headway_s"] / leaders["v_hdv"],
        np.nan
    )

    leaders = leaders[[
        "hdv_id", "time",
        "leader_id" if "leader_id" in leaders.columns else "agent_id",
        "leader_type",
        "headway_s",
        "headway_t",
        "delta_v"
    ]]

    leaders = leaders.rename(columns={"agent_id": "leader_id"})

    leader_results.append(leaders)

# ===============================
# Combine windows
# ===============================
leaders_all = pd.concat(leader_results, ignore_index=True)

leaders_all = leaders_all.groupby(
    ["hdv_id", "time"], as_index=False
).first()

# ===============================
# Final dataset (renamed columns preserved)
# ===============================
final_df = hdv_base.merge(
    leaders_all,
    on=["hdv_id", "time"],
    how="left"
)

final_df.to_csv("TGSIM_I-395_DM_Dataset.csv", index=False)

print("Done.")
print(final_df.columns)
print("Rows:", len(final_df))


Done.
Index(['hdv_id', 'time', 'lane_hdv', 'v_hdv', 'xloc_hdv', 'yloc_hdv', 'vx_hdv',
       'vy_hdv', 'a_hdv', 'ax_hdv', 'ay_hdv', 'jerk_hdv', 'leader_id',
       'leader_type', 'headway_s', 'headway_t', 'delta_v'],
      dtype='object')
Rows: 4214543


## Adding Context Variables

### Distance to nearest pedestrian

In [2]:
import pandas as pd

# Load main dataset
dm = pd.read_csv("TGSIM_I-395_DM_Dataset.csv")
dm_3000 = dm[dm['time']<3000] # We're trying the first 3000 seconds

# No peds in this dataset, we'll assign a value far greater than max d_ped in FB
dm_3000["d_ped"] = 500

# Save
dm_3000.to_csv("TGSIM_I-395_DM_Dataset_with_dped.csv", index=False)

C:\Users\msela\AppData\Local\Temp\ipykernel_20768\3253517144.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dm_3000["d_ped"] = 500


### Distance to stop line

In [3]:
import pandas as pd
import numpy as np

dm = pd.read_csv("TGSIM_I-395_DM_Dataset_with_dped.csv")

# No stop lines here either, we'll assign a value far greater than max d_stop in FB
dm["d_stop"] = 500

dm.to_csv("TGSIM_I-395_DM_Dataset_with_dped_dstop.csv", index=False)

### Local density and average speed

In [4]:
import pandas as pd

# --------------------------------------------------
# 1. Load main dataset
# --------------------------------------------------
dm = pd.read_csv("TGSIM_I-395_DM_Dataset_with_dped_dstop.csv")
dm["time"] = (dm["time"] * 10).round().astype(int) / 10

# --------------------------------------------------
# 2. Load trajectory dataset once
# --------------------------------------------------
traj = pd.read_csv("Transformed_TGSIM_I-395_smoothed.csv")
traj["time"] = (traj["time"] * 10).round().astype(int) / 10

motorized_types = {1, 2, 3, 4}
traj_motorized = traj[traj["type_most_common"].isin(motorized_types)][
    ["id", "time", "speed_kf_smooth"]
].rename(columns={
    "id": "agent_id",
    "speed_kf_smooth": "agent_speed"
})

eligible_zones = {
    "Wide Fwd",
    "Main Fwd",
    "Narrow Fwd",
    "Side Fwd L",
    "Side Fwd R"
}

all_density = []
all_avg_v = []

# --------------------------------------------------
# 3. Single pass through windows
# --------------------------------------------------
for window_id in range(3):

    print(f"Processing window {window_id}...")

    detections = pd.read_csv(
        f"HDV_Detection_Details_I-395_window_{window_id}.csv"
    )

    detections["time"] = (
        detections["time"] * 10
    ).round().astype(int) / 10

    # Remove duplicate overlaps
    detections = detections.drop_duplicates(
        subset=["hdv_id", "time", "agent_id"]
    )

    # -------- Density --------
    density_df = (
        detections
        .groupby(["hdv_id", "time"])["agent_id"]
        .nunique()
        .reset_index()
        .rename(columns={"agent_id": "density"})
    )
    all_density.append(density_df)

    # -------- avg_v --------
    det_forward = detections[detections["zone"].isin(eligible_zones)]

    det_forward = det_forward.merge(
        traj_motorized,
        on=["agent_id", "time"],
        how="inner"
    )

    avg_v_df = (
        det_forward
        .groupby(["hdv_id", "time"])["agent_speed"]
        .mean()
        .reset_index()
        .rename(columns={"agent_speed": "avg_v"})
    )

    all_avg_v.append(avg_v_df)

print("All windows processed.")

# --------------------------------------------------
# 4. Combine results
# --------------------------------------------------
density_full = (
    pd.concat(all_density, ignore_index=True)
    .drop_duplicates(["hdv_id", "time"])
)

avg_v_full = (
    pd.concat(all_avg_v, ignore_index=True)
    .drop_duplicates(["hdv_id", "time"])
)

# --------------------------------------------------
# 5. Merge once
# --------------------------------------------------
dm = dm.merge(density_full, on=["hdv_id", "time"], how="left")
dm = dm.merge(avg_v_full, on=["hdv_id", "time"], how="left")

dm["density"] = dm["density"].fillna(0).astype(int)
dm["avg_v"] = dm["avg_v"].fillna(0)

# --------------------------------------------------
# 6. Save
# --------------------------------------------------
dm.to_csv("TGSIM_I-395_DM_Final_Dataset.csv", index=False)

print("Done.")

Processing window 0...
Processing window 1...
Processing window 2...
All windows processed.
Done.


## Handling NaNs

In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("TGSIM_I-395_DM_Final_Dataset.csv")

# Replace inf first
df = df.replace([np.inf, -np.inf], np.nan)

# Behavioral reference max
max_headway = df["headway_s"].quantile(0.999)
print(f"max_headway = {max_headway}m")
# BEHAVIORAL VARIABLES

# No leader → delta_v = 0
df["delta_v"] = df["delta_v"].fillna(0)

# No leader → headway = large
df["headway_s"] = df["headway_s"].fillna(max_headway)

print("Remaining NaNs:")
print(df[["delta_v", "headway_s", "a_hdv",
          "d_ped", "d_stop", "density"]].isna().sum())
df.to_csv("TGSIM_I-395_DM_Final_Dataset.csv", index=False)

max_headway = 113.65791482617075m
Remaining NaNs:
delta_v      0
headway_s    0
a_hdv        0
d_ped        0
d_stop       0
density      0
dtype: int64


In [9]:
import pandas as pd

# Load
fb = pd.read_csv("TGSIM_FB_DM_Final_Dataset.csv")
i395 = pd.read_csv("TGSIM_I-395_DM_Final_Dataset.csv")

TIME_STEP = 0.1
MAX_ALLOWED_GAP = 0.3
GAP_THRESHOLD = TIME_STEP + MAX_ALLOWED_GAP  # 0.4
MIN_DURATION = 30


def filter_valid_trajectories(df, dataset_name):
    
    valid_ids = []

    for hdv_id, group in df.groupby("hdv_id"):
        group = group.sort_values("time")

        duration = group["time"].max() - group["time"].min()
        if duration < MIN_DURATION:
            continue

        time_diff = group["time"].diff().dropna()
        max_gap = time_diff.max()

        if pd.isna(max_gap) or max_gap <= GAP_THRESHOLD:
            valid_ids.append(hdv_id)

    filtered_df = df[df["hdv_id"].isin(valid_ids)].copy()
    filtered_df["dataset"] = dataset_name
    
    return filtered_df


fb_filtered = filter_valid_trajectories(fb, "FB")
i395_filtered = filter_valid_trajectories(i395, "I395")

final_dataset = pd.concat([fb_filtered, i395_filtered], ignore_index=True)

print("Final dataset shape:", final_dataset.shape)
print("Unique vehicles:", final_dataset["hdv_id"].nunique())

# Optional save
final_dataset.to_csv("TGSIM_Combined_30s_Cleaned_3000_0.999.csv", index=False)

Final dataset shape: (3200397, 22)
Unique vehicles: 4360
